# stellantis-sales-audit-engine
**Author:** [Joel Perez Guerrero]  
**Tools:** Python, pgAdmin, Pandas.  
-In this notebook we perform data extraction from PostgreSQL.

1. Upload DB from postgresql

In [1]:
# DATA BASE CONNECTION SETTINGS
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv('../.env')

def get_connection():
    try:
        # 2. Get credentials
        user = os.getenv('DB_USER')
        password = os.getenv('DB_PASS') 
        host = os.getenv('DB_HOST')
        port = os.getenv('DB_PORT')
        db = os.getenv('DB_NAME')

        missing_vars = [var for var, val in {
            'DB_USER': user, 'DB_PASS': password, 
            'DB_HOST': host, 'DB_PORT': port, 'DB_NAME': db
        }.items() if val is None]

        if missing_vars:
            raise ValueError(f"Faltan variables en el .env: {', '.join(missing_vars)}")

        #Create connection engine
        url_conexion = f'postgresql://{user}:{password}@{host}:{port}/{db}'
        engine = create_engine(url_conexion)
        
        # Conection test
        with engine.connect() as conn:
            print(f"✅ Conexión establecida con éxito a la base de datos: {db}")
        
        return engine

    except Exception as e:
        print(f"❌ Error al conectar: {e}")
        return None

# Execute connection
engine = get_connection()

#if engine is None:
    # Esto detendrá la ejecución del notebook si falla la conexión
    #raise Exception("No se puede continuar sin conexión a la base de datos.")

✅ Conexión establecida con éxito a la base de datos: stellantis_db


2. We run query to join tables and define dataframe.
- We need to clasify outliers using sales amount and brands. So, fact_ventas and dim_vehiculos are joined.
- Create "raw_data" file

In [2]:
#FROM DB LOAD DATA AS DATAFRAME

query = """

SELECT *
FROM fact_ventas_cl v
JOIN dim_vehiculos_cl u
ON v.id_vehiculo = u.id_vehiculo
ORDER BY v.id_transaccion
;

"""
if engine:
    # Read raw data using query
    df = pd.read_sql(query, engine)
    df= df.loc[:, ~df.columns.duplicated()] #Drop duplicated columns if exist
    df.to_parquet('../data/raw_data.parquet', index=False)
    print(f"📊 Dataset cargado: {df.shape[0]} filas extraídas.")
    display(df.head(10))
else:
    print("❌ No se pudo cargar data frame de la base de datos.")


📊 Dataset cargado: 536321 filas extraídas.


,id_transaccion,id_vehiculo,id_ubicacion,id_cliente,fecha,unidades,venta_bruta_sn,modelo,marcas_cl
0,ST-2000000,1,1,2536,2024-12-01,1,679237.0,Argo,Fiat
1,ST-2000001,2,2,4867,2022-09-22,1,419582.0,1500,RAM
2,ST-2000002,3,1,4669,2024-12-01,1,834373.0,Grand Cherokee,Jeep
3,ST-2000003,4,3,899,2024-04-27,1,945477.0,ProMaster,RAM
4,ST-2000004,5,4,2471,2025-11-13,1,433661.0,Wrangler,Jeep
5,ST-2000005,6,4,6717,2022-01-06,1,428597.0,3008,Peugeot
6,ST-2000006,7,1,3359,2024-05-07,1,1065849.0,Giulia,Alfa Romeo
7,ST-2000007,1,4,3313,2024-06-07,1,375204.0,Argo,Fiat
8,ST-2000008,8,1,2377,2023-06-16,1,468735.0,700,RAM
9,ST-2000009,4,2,1750,2022-05-16,1,802080.0,ProMaster,RAM
